# 한국 배당성장주 백테스트 & 필터 검증 (Google Colab + PDF 리포트)

외부 API 키 없이 Google Colab에서 **`런타임 → 모두 실행`** 한 번으로 다음을 자동 수행합니다.

1. **한글 폰트 & PDF 엔진 자동 설치** (`NanumGothic` + `WeasyPrint`)
2. **공개 시점고정(PIT) 재무·배당 데이터 로드** (한국 상장사 374개사 `FY2018~2024` + `FY2017~2025` 보정 실제 DPS + 국고채 10년물)
3. **일별 수정주가·거래대금·KOSPI 실시간 수집** (네이버 `fchart` → `FinanceDataReader` → `pykrx` 3중 폴백)
4. **전략 A(현행 규칙) vs 전략 B(함정 보완안) 동시 백테스트 + 5개 필터 개별 분해(Ablation Test)**
5. **모바일·PC 원클릭 열람용 고화질 PDF 리포트(`backtest_report_YYYY-MM-DD.pdf`) 자동 생성 · Drive 저장 · 다운로드**

> **한계 고지:** 374개사 표본은 현재 상장 생존 기업 기준(생존 편향 가능성)이며, KOSPI 벤치마크는 배당 미포함 가격지수(PR)입니다.

In [ ]:
# ==============================================================================
# 1. 환경 설정, 패키지 설치 및 한글 폰트 자동 구성
# ==============================================================================
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[환경] Google Colab 감지 -> 한글 폰트(fonts-nanum) 및 FinanceDataReader 설치 중...")
    subprocess.run(["apt-get", "-qq", "update"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum", "libpango-1.0-0", "libpangoft2-1.0-0"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "finance-datareader", "weasyprint"], check=False)

import re, gzip, json, math, time, statistics
from pathlib import Path
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mtick

# 한글 폰트 자동 탐색 및 즉시 등록 (Colab 런타임 재시작 없이 바로 적용)
def setup_korean_font():
    nanum_paths = [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf",
    ]
    chosen = None
    for p in nanum_paths:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            if chosen is None:
                chosen = fm.FontProperties(fname=p).get_name()

    if chosen is None:
        available = {f.name for f in fm.fontManager.ttflist}
        for cand in ["NanumGothic", "AppleGothic", "Malgun Gothic", "Noto Sans CJK KR", "DejaVu Sans"]:
            if cand in available:
                chosen = cand
                break

    if chosen:
        plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except Exception:
        pass
    if chosen:
        plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 120
    return chosen

FONT_NAME = setup_korean_font()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print(f"[완료] 폰트 설정: {FONT_NAME} | pandas {pd.__version__} | numpy {np.__version__}")
try:
    from IPython.display import display, HTML
except ImportError:
    display = print
    HTML = lambda x: x

In [ ]:
# ==============================================================================
# 2. 백테스트 파라미터 설정 (필요에 따라 자유롭게 수정 후 모두 실행)
# ==============================================================================
CONFIG = {
    # 백테스트 기준 사업연도 범위 (FY2018 -> 2019년 4월 출발: 2020 코로나·2022 하락·2026 급락 3대 폭락장 모두 포함)
    "START_FY": 2018,
    "END_FY": 2024,            # FY2024 -> 2025년 4월 리밸런싱 후 최신 거래일까지 보유

    # 포트폴리오 운용 가정
    "TOP_N": 10,               # 리밸런싱 시 상위 N종목 동일가중 편입
    "INITIAL_CAPITAL": 100_000_000,  # 초기 투자금 1억 원
    "FEE_RATE": 0.0025,        # 매매 편도 비용 0.25% (수수료 + 거래세 근사)
    "DIV_TAX_RATE": 0.154,     # 배당소득세 15.4%
    "REINVEST_DIVIDENDS": True,# 세후 배당금 재투자(DRIP) 여부

    # 품질 판정 결측 허용 (초기 연도 일부 항목 결측 시 최대 1개까지 허용; 엄격 모드는 0)
    "ALLOW_NA_COUNT": 1,

    # 전략 B (배당 함정 보완안) 추가 임계값
    "B_MIN_YIELD": 0.02,                # 최소 배당수익률 2.0%
    "B_MAX_YIELD": 0.12,                # 최대 배당수익률 12.0% (폭락형 초고배당 함정 차단)
    "B_MIN_DPS_CAGR_5Y": 0.03,          # 정기 DPS 5년 연평균 성장률 >= 3.0%
    "B_MIN_TURNOVER_60D": 1_000_000_000,# 60일 일평균 거래대금 >= 10억 원
    "B_MAX_DD_52W_WITH_EPS_DROP": -0.35,# 52주 고점 대비 -35% 이하 & 최근 EPS 역성장 동시 시 제외
    "B_VAL_CAP": 0.50,                  # 밸류에이션 개별 지표 괴리율 ±50% 캡(윈저라이징)

    # 공개 데이터 저장소 (로컬 data/ 우선, 없으면 GitHub Public Raw URL에서 자동 다운로드)
    "GITHUB_REPO": "khs0221/Common_Everything",
    "GITHUB_BRANCHES": ["feat/dividend-backtest", "main", "copilot/ai-coding-curriculum"],
    "DATA_SUBPATH": "dividend-backtest/data",
    "CACHE_DIR": "./_cache_dividend_backtest",
    "SAVE_REPORT_TO_DRIVE": True,
    "DRIVE_REPORT_DIR": "/content/drive/MyDrive/Colab Notebooks/dividend_backtest/reports",
}

print("[설정] 백테스트 기간: FY{START_FY}~FY{END_FY} (리밸런싱 {r0}-04 ~ {r1}-04 -> 현재)".format(
    START_FY=CONFIG["START_FY"], END_FY=CONFIG["END_FY"],
    r0=CONFIG["START_FY"] + 1, r1=CONFIG["END_FY"] + 1
))
print(f"[설정] 상위 {CONFIG['TOP_N']}종목 동일가중 | 초기자금 {CONFIG['INITIAL_CAPITAL']:,.0f}원 | "
      f"편도비용 {CONFIG['FEE_RATE']:.2%} | 배당세 {CONFIG['DIV_TAX_RATE']:.1%} | DRIP={CONFIG['REINVEST_DIVIDENDS']}")

In [ ]:
# ==============================================================================
# 3. 공개 Point-in-Time 재무·배당 데이터 및 국고채 10년물 로드
# ==============================================================================
CACHE_DIR = Path(CONFIG["CACHE_DIR"])
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def ensure_data_file(filename: str) -> Path:
    # 1) 로컬 실행 시 상대 경로 탐색
    for local_cand in [Path("../data") / filename, Path("data") / filename, Path("dividend-backtest/data") / filename]:
        if local_cand.exists():
            return local_cand

    # 2) 캐시 확인
    cached = CACHE_DIR / filename
    if cached.exists() and cached.stat().st_size > 0:
        return cached

    # 3) GitHub Public Raw URL 자동 다운로드 (브랜치 순차 탐색)
    last_err = None
    for br in CONFIG["GITHUB_BRANCHES"]:
        url = f"https://raw.githubusercontent.com/{CONFIG['GITHUB_REPO']}/{br}/{CONFIG['DATA_SUBPATH']}/{filename}"
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200 and len(r.content) > 0:
                cached.write_bytes(r.content)
                print(f"  [다운로드 완료] {filename} ({len(r.content):,} bytes, branch={br})")
                return cached
            last_err = f"HTTP {r.status_code} ({url})"
        except Exception as e:
            last_err = str(e)
    raise RuntimeError(f"데이터 파일({filename})을 가져오지 못했습니다: {last_err}")

pit_path = ensure_data_file("kr_fundamentals_pit.jsonl.gz")
rf_path = ensure_data_file("kr_treasury_10y_annual.csv")
kospi_snap_path = ensure_data_file("kospi_snapshots.json")

with gzip.open(pit_path, "rt", encoding="utf-8") as f:
    PIT_RECORDS = [json.loads(line) for line in f if line.strip()]

rf_df = pd.read_csv(rf_path)
RF_MAP = {int(row["연도"]): float(row["국고채10년_연평균_퍼센트"]) / 100.0 for _, row in rf_df.iterrows()}

with open(kospi_snap_path, encoding="utf-8") as f:
    KOSPI_SNAP = json.load(f)

mkt_counts = pd.Series([r["market"] for r in PIT_RECORDS]).value_counts().to_dict()
path_counts = pd.Series([r["path"] for r in PIT_RECORDS]).value_counts().to_dict()
print(f"[로드 완료] 상장사 {len(PIT_RECORDS)}개 | 시장={mkt_counts} | 업종경로={path_counts}")
print(f"[로드 완료] 국고채 10년물 연평균 금리: {min(RF_MAP)}~{max(RF_MAP)}년 (평균 {np.mean(list(RF_MAP.values())):.2%})")

In [ ]:
# ==============================================================================
# 4. 배당성장주 정량 판정 엔진 (1~9번 판정 + 전략 A/B 1차 품질 풀 산출)
# ==============================================================================
ERP = 0.05   # 주식위험프리미엄 5.0% 고정
TAX = 0.24   # 법인세율 24% 고정
REG_CAP = {
    "bank": ("cet1", 8.0, 2.0),
    "insurance": ("kics", 100.0, 50.0),
    "financial_holding": ("fh_capital", 100.0, 30.0),
    "securities": ("ncr", 100.0, 50.0),
}

def _f(v):
    if v is None:
        return None
    try:
        x = float(v)
        return None if (math.isnan(x) or math.isinf(x)) else x
    except (TypeError, ValueError):
        return None

def get_val(frame: dict, col: str, y: int):
    d = frame.get(col)
    return _f(d.get(str(y))) if isinstance(d, dict) else None

def avg3(frame: dict, col: str, end: int):
    vals = [v for y in (end - 2, end - 1, end) if (v := get_val(frame, col, y)) is not None]
    return sum(vals) / len(vals) if len(vals) >= 2 else None

def cagr_3avg(frame: dict, col: str, end: int, span: int):
    a, b = avg3(frame, col, end - span), avg3(frame, col, end)
    if a is None or a <= 0 or b is None:
        return None
    return -1.0 if b <= 0 else (b / a) ** (1.0 / span) - 1.0

def total_return_3y(frame: dict, end: int):
    W = (end - 2, end - 1, end)
    decl, buy, canc, acq, e3 = [], [], [], [], []
    for y in W:
        dps = get_val(frame, "dps_common", y)
        d = get_val(frame, "div_total_reported", y)
        if d is None and dps == 0:
            d = 0.0
        decl.append(d)
        b = get_val(frame, "treasury_purchase", y)
        has_fs = bool((frame.get("_fs_src") or {}).get(str(y)))
        buy.append(abs(b) if b is not None else (0.0 if has_fs else None))
        canc.append(get_val(frame, "ts_cancelled_common", y))
        acq.append(get_val(frame, "ts_acquired_common", y))
        nip = get_val(frame, "net_income_parent", y)
        nr = get_val(frame, "ni_reported", y)
        ni = get_val(frame, "net_income", y)
        e3.append(nip if nip is not None else (nr if nr is not None else ni))
    if any(v is None for v in decl) or any(v is None for v in e3):
        return None
    e_sum, d_sum = sum(e3), sum(decl)
    if any(v is None for v in buy):
        if any(v is None for v in canc) or sum(canc) > 0:
            return None
        num = d_sum
    else:
        b_sum = sum(buy)
        if b_sum == 0:
            s = 0.0
        else:
            k = sum(v for v in canc if v is not None)
            a = sum(v for v in acq if v is not None)
            s = min(1.0, k / a) if a > 0 else (1.0 if k > 0 else 0.0)
        num = d_sum + b_sum * s
    if e_sum <= 0:
        return {"tr": -1.0, "payout3": None, "num": num, "e3": e_sum}
    return {"tr": num / e_sum * 100.0, "payout3": d_sum / e_sum * 100.0, "num": num, "e3": e_sum}

def evaluate_company_pit(rec: dict, Y: int) -> dict | None:
    '''기준 사업연도 Y의 시점고정(Point-in-Time) 스냅샷으로 1~9번 판정 (미래 정보 누출 차단).'''
    frame = (rec.get("snapshots") or {}).get(str(Y))
    if not frame:
        return None
    price_info = rec.get("price") or {}
    ye_px = {int(k): _f(v) for k, v in (price_info.get("year_end") or {}).items()}
    t_px = _f((price_info.get("t_close") or {}).get(str(Y)))
    if not t_px or t_px <= 0:
        return None

    path = rec.get("path", "general")
    cap = rec.get("capital") or {}
    dps_y = get_val(frame, "dps_common", Y)
    if dps_y is None or dps_y <= 0:
        return None

    # 1번 배당 이력 (과거 시점 백테스트 자료 창 보정: 최근 5년 연속 지급·무삭감·DPS 5년 성장 > 0)
    paid_5y = all((get_val(frame, "dps_common", y) or 0) > 0 for y in range(Y - 4, Y + 1))
    cuts_5y = 0
    for y in range(Y - 4, Y + 1):
        a, b = get_val(frame, "dps_common", y - 1), get_val(frame, "dps_common", y)
        if a is not None and b is not None and b < a - min(0.5, abs(a) * 1e-6):
            cuts_5y += 1
    dps_g5 = cagr_3avg(frame, "dps_common", Y, 5)
    if dps_g5 is None and Y == 2018:   # FY2018은 DART 배당 개시(2013) 제약으로 3년(2013~15 vs 2016~18) 성장률 보완
        dps_g5 = cagr_3avg(frame, "dps_common", Y, 3)
    if not paid_5y or cuts_5y >= 2:
        v1 = "FAIL"
    elif cuts_5y == 0 and dps_g5 is not None and dps_g5 > 0:
        v1 = "PASS"
    else:
        v1 = "WARN"

    # 2번 EPS 성장 (3년 배당성향 30% 미만 시 DPS 초과 성장 면제)
    eps_g5 = cagr_3avg(frame, "eps_reported", Y, 5)
    if eps_g5 is None and Y == 2018:
        eps_g5 = cagr_3avg(frame, "eps_reported", Y, 3)
    tr = total_return_3y(frame, Y)
    p3 = tr["payout3"] if tr else None
    if eps_g5 is None or dps_g5 is None:
        v2 = "NA"
    elif eps_g5 < 0 and dps_g5 > 0:
        v2 = "FAIL"
    elif eps_g5 > 0 and (dps_g5 <= eps_g5 + 0.03 or (p3 is not None and p3 < 30.0)):
        v2 = "PASS"
    else:
        v2 = "WARN"

    # 3번 FCF(일반) / 순이익(금융) 꾸준함
    col3 = "fcf" if path == "general" else "net_income_parent"
    vals3 = [get_val(frame, col3, y) for y in range(Y - 9, Y + 1)]
    if path != "general":
        vals3 = [v if v is not None else get_val(frame, "ni_reported", y) for v, y in zip(vals3, range(Y - 9, Y + 1))]
    known3 = [v for v in vals3 if v is not None]
    neg3, pos3 = sum(1 for v in known3 if v < 0), sum(1 for v in known3 if v > 0)
    if neg3 >= 3:
        v3 = "FAIL"
    elif len(known3) >= 4 and pos3 / len(known3) >= 0.8:
        v3 = "PASS"
    elif len(known3) >= 4:
        v3 = "WARN"
    else:
        v3 = "NA"

    # 4번 자본효율 (일반 ROIC-WACC >= 3%p, 금융 ROE-COE >= 1%p)
    spreads = []
    for y in range(Y - 4, Y + 1):
        rf = RF_MAP.get(y, 0.03)
        coe = rf + ERP
        if path == "general":
            r_val = get_val(frame, "roic", y)
            if r_val is None:
                continue
            px_y = ye_px.get(y)
            sh_y = (get_val(frame, "outstanding_common", y) or 0) + (get_val(frame, "outstanding_pref", y) or 0)
            debt_y = get_val(frame, "debt", y) or 0.0
            mcap_y = px_y * sh_y if (px_y and sh_y) else None
            if mcap_y and mcap_y > 0:
                tot = mcap_y + debt_y
                int_y = get_val(frame, "interest_expense", y) or get_val(frame, "interest_paid", y)
                kd = (min(0.15, abs(int_y) / debt_y) if (debt_y > 0 and int_y and int_y > 0) else 0.04) * (1 - TAX)
                wacc = coe * (mcap_y / tot) + kd * (debt_y / tot)
            else:
                wacc = coe
            spreads.append((y, r_val - wacc))
        else:
            r_val = get_val(frame, "roe", y)
            if r_val is not None:
                spreads.append((y, r_val - coe))
    pass_thr, fail_thr = (0.03, 0.0) if path == "general" else (0.01, -0.01)
    avg5_sp = sum(sp for _, sp in spreads) / len(spreads) if len(spreads) >= 3 else None
    last3_sp = [sp for y, sp in spreads if y >= Y - 2]
    if len(last3_sp) == 3 and all(sp < fail_thr for sp in last3_sp):
        v4 = "FAIL"
    elif avg5_sp is not None:
        rec2_sp = [sp for y, sp in spreads if y >= Y - 1]
        v4 = "PASS" if (avg5_sp >= pass_thr and (not rec2_sp or sum(rec2_sp) / len(rec2_sp) >= avg5_sp - 0.03)) else "WARN"
    else:
        v4 = "NA"

    # 5번 주주환원 수준 (3년 총환원율 >= 30% & 배당성향 <= 60%, 금융 상한 추가)
    if tr is None:
        v5 = "NA"
    elif tr["e3"] <= 0 or tr["tr"] < 20.0 or (p3 is not None and p3 > 75.0) or (path != "general" and tr["tr"] > 100.0):
        v5 = "FAIL"
    elif tr["tr"] >= 30.0 and (p3 is not None and p3 <= 60.0) and (path == "general" or tr["tr"] <= 66.7):
        v5 = "PASS"
    else:
        v5 = "WARN"

    # 6번 자사주·희석 (FY2019는 DART 주식총수 개시연도 제약으로 Y-4 보완 허용)
    sh_now = get_val(frame, "outstanding_common", Y)
    sh_base = get_val(frame, "outstanding_common", Y - 5) or (get_val(frame, "outstanding_common", 2015) if Y in (2018, 2019) else None)
    if sh_now is None or not sh_base:
        v6 = "NA"
    else:
        chg = sh_now / sh_base - 1.0
        canc5 = sum(get_val(frame, "ts_cancelled_common", y) or 0.0 for y in range(Y - 4, Y + 1))
        acq5 = sum(get_val(frame, "ts_acquired_common", y) or 0.0 for y in range(Y - 4, Y + 1))
        if chg >= 0.03:
            v6 = "FAIL"
        elif (chg < -0.01 and canc5 > 0) or (-0.01 <= chg <= 0.01 and (acq5 == 0 or canc5 >= acq5)):
            v6 = "PASS"
        else:
            v6 = "WARN"

    # 7번 환원 감당 (일반 경로만 품질 종합에 산입)
    if path != "general":
        v7 = "PASS"
    else:
        fcfs = [get_val(frame, "fcf", y) for y in (Y - 2, Y - 1, Y)]
        if tr is None or any(v is None for v in fcfs):
            v7 = "NA"
        else:
            fcf3 = sum(fcfs)
            if fcf3 <= 0:
                v7 = "FAIL" if tr["num"] > 0 else "PASS"
            else:
                cov = tr["num"] / fcf3
                v7 = "FAIL" if cov > 1.0 else ("WARN" if cov > 2 / 3 else "PASS")

    # 8번 재무건전성 (일반: 순부채/EBITDA & ICR, 금융: 규제자본비율)
    if path == "general":
        nd, nde = get_val(frame, "net_debt", Y), get_val(frame, "net_debt_to_ebitda", Y)
        icr, eb = get_val(frame, "interest_coverage", Y), get_val(frame, "ebitda", Y)
        if nd is not None and nd <= 0:
            v8 = "WARN" if (eb is not None and eb <= 0) else "PASS"
        elif (icr is not None and icr < 3.0) or (nde is not None and nde > 3.0):
            v8 = "FAIL"
        elif nde is not None and icr is not None:
            v8 = "PASS" if (nde < 2.0 and icr >= 5.0) else "WARN"
        else:
            v8 = "NA"
    else:
        mk, req, margin = REG_CAP.get(path, ("cet1", 8.0, 2.0))
        cval = _f((cap.get(str(Y)) or {}).get(mk))
        v8 = "NA" if cval is None else ("PASS" if cval >= req + margin else ("WARN" if cval >= req else "FAIL"))

    v_pairs = [(1, v1), (2, v2), (3, v3), (4, v4), (5, v5), (6, v6), (8, v8)] if path != "general" else [(1, v1), (2, v2), (3, v3), (4, v4), (5, v5), (6, v6), (7, v7), (8, v8)]
    verdicts = [v for _, v in v_pairs]
    fail_items = [i for i, v in v_pairs if v == "FAIL"]
    n_fail, n_warn, n_na = verdicts.count("FAIL"), verdicts.count("WARN"), verdicts.count("NA")
    qual_pass = (n_fail == 0 and n_warn <= 2 and n_na <= CONFIG["ALLOW_NA_COUNT"])

    # 9번 밸류에이션 (후행 FY 기준 vs 과거 연말 중앙값)
    eps_y = get_val(frame, "eps_reported", Y)
    eps_prev = get_val(frame, "eps_reported", Y - 1)
    eq_y = get_val(frame, "equity_parent", Y) or get_val(frame, "equity", Y)
    sh_tot = (get_val(frame, "outstanding_common", Y) or 0) + (get_val(frame, "outstanding_pref", Y) or 0)
    bps_y = eq_y / sh_tot if (eq_y and sh_tot > 0) else None

    cur_per = t_px / eps_y if (eps_y and eps_y > 0) else None
    cur_pbr = t_px / bps_y if (bps_y and bps_y > 0) else None
    cur_yld = dps_y / t_px if (dps_y and dps_y > 0) else None

    hist_per, hist_pbr, hist_yld = [], [], []
    for y in range(Y - 9, Y + 1):
        px = ye_px.get(y)
        if not px or px <= 0:
            continue
        e, d = get_val(frame, "eps_reported", y), get_val(frame, "dps_common", y)
        eq = get_val(frame, "equity_parent", y) or get_val(frame, "equity", y)
        sh = (get_val(frame, "outstanding_common", y) or 0) + (get_val(frame, "outstanding_pref", y) or 0)
        b = eq / sh if (eq and sh > 0) else None
        if e and e > 0: hist_per.append(px / e)
        if b and b > 0: hist_pbr.append(px / b)
        if d and d > 0: hist_yld.append(d / px)

    cap_lim = CONFIG["B_VAL_CAP"]
    devs_raw, devs_cap = [], []
    for cur_v, hist_v, higher_is_cheap in ((cur_per, hist_per, False), (cur_pbr, hist_pbr, False), (cur_yld, hist_yld, True)):
        if cur_v is not None and len(hist_v) >= 4:
            med = statistics.median(hist_v)
            if med > 0:
                dev = (cur_v / med - 1.0) if higher_is_cheap else -(cur_v / med - 1.0)
                devs_raw.append(dev)
                devs_cap.append(max(-cap_lim, min(cap_lim, dev)))

    val_raw = sum(devs_raw) / len(devs_raw) if len(devs_raw) >= 2 else None
    val_cap = sum(devs_cap) / len(devs_cap) if len(devs_cap) >= 2 else None

    return {
        "fy": Y, "rebal_year": Y + 1, "ticker": rec["ticker"], "name": rec["name"],
        "market": rec.get("market", "KOSPI"), "path": path,
        "qual_pass": qual_pass, "n_fail": n_fail, "fail_items": fail_items, "n_warn": n_warn, "n_na": n_na,
        "val_raw": val_raw, "val_cap": val_cap,
        "dps_g5": dps_g5, "eps_g5": eps_g5, "cur_yld": cur_yld, "dps_y": dps_y, "dps_y_adj": (_f((rec.get("act_dps") or {}).get(str(Y))) or dps_y),
        "eps_down": bool(eps_y is not None and eps_prev is not None and eps_y < eps_prev),
        "spread": avg5_sp if avg5_sp is not None else -9.0,
        "tr_3y": tr["tr"] if tr else None,
        "payout_3y": p3,
    }

# 전 연도(FY2019~FY2024) 1차 품질 통과 후보군 산출
YEARLY_POOL = {}
NEEDED_TICKERS = set()
for fy in range(CONFIG["START_FY"], CONFIG["END_FY"] + 1):
    rows = [x for r in PIT_RECORDS if (x := evaluate_company_pit(r, fy)) is not None]
    pool = [x for x in rows if x["qual_pass"] and x["val_raw"] is not None]
    YEARLY_POOL[fy] = pool
    NEEDED_TICKERS.update(x["ticker"] for x in pool)
    print(f"  FY{fy} (리밸런싱 {fy+1}-04): 품질 통과 후보 {len(pool):2d}종목")

print(f"\n[완료] 일별 시세 수집 대상 고유 종목 수: {len(NEEDED_TICKERS)}개 (+ KOSPI 지수)")
PIT_REC_MAP = {r["ticker"]: r for r in PIT_RECORDS}

In [ ]:
# ==============================================================================
# 5. 한국 증시 일별 수정주가·거래대금·KOSPI 실시간 수집 및 전략 A/B 최종 종목 선정
# ==============================================================================
_FCHART_URL = "https://fchart.stock.naver.com/sise.nhn"
_ITEM_RE = re.compile(r'<item data="([^"]+)"')
_HEADERS = {"User-Agent": "Mozilla/5.0 (dividend-backtest-colab)"}

def fetch_naver_ohlcv(symbol: str, count: int = 3200) -> pd.DataFrame:
    '''네이버 금융 fchart API (무인증, 수정주가 일봉). symbol: 6자리 티커 또는 "KOSPI".'''
    last_err = None
    for attempt in range(3):
        try:
            r = requests.get(
                _FCHART_URL,
                params={"symbol": symbol, "timeframe": "day", "count": count, "requestType": 0},
                headers=_HEADERS, timeout=20,
            )
            r.raise_for_status()
            rows = []
            for m in _ITEM_RE.finditer(r.content.decode("euc-kr", errors="ignore")):
                d, o, h, l, c, v = m.group(1).split("|")
                rows.append((pd.Timestamp(d), float(o), float(h), float(l), float(c), float(v)))
            df = pd.DataFrame(rows, columns=["date", "open", "high", "low", "close", "volume"]).set_index("date")
            df = df[df["close"] > 0].sort_index()
            if not df.empty:
                return df
            last_err = ValueError("empty naver response")
        except Exception as e:
            last_err = e
            time.sleep(0.5 * (attempt + 1))
    raise RuntimeError(f"Naver fchart 실패({symbol}): {last_err}")

def fetch_daily_ohlcv_resilient(symbol: str, start: str = "2017-01-01") -> pd.DataFrame:
    '''1순위: 네이버 fchart -> 2순위: FinanceDataReader -> 3순위: pykrx 폴백.'''
    try:
        df = fetch_naver_ohlcv(symbol)
        return df.loc[df.index >= pd.Timestamp(start)]
    except Exception:
        pass

    try:
        import FinanceDataReader as fdr
        fdr_sym = "KS11" if symbol == "KOSPI" else symbol
        df = fdr.DataReader(fdr_sym, start)
        col_map = {c: c.lower() for c in df.columns}
        df = df.rename(columns=col_map)
        for col in ["open", "high", "low", "close", "volume"]:
            if col not in df.columns:
                df[col] = df["close"] if col != "volume" else 0.0
        df = df[["open", "high", "low", "close", "volume"]]
        df = df[df["close"] > 0].sort_index()
        if not df.empty:
            return df
    except Exception:
        pass

    try:
        from pykrx import stock as krx_stock
        end_str = datetime.today().strftime("%Y%m%d")
        start_str = start.replace("-", "")
        if symbol == "KOSPI":
            df = krx_stock.get_index_ohlcv(start_str, end_str, "1001")
            df = df.rename(columns={"시가": "open", "고가": "high", "저가": "low", "종가": "close", "거래량": "volume"})
        else:
            df = krx_stock.get_market_ohlcv(start_str, end_str, symbol, adjusted=True)
            df = df.rename(columns={"시가": "open", "고가": "high", "저가": "low", "종가": "close", "거래량": "volume"})
        df = df[["open", "high", "low", "close", "volume"]]
        df = df[df["close"] > 0].sort_index()
        if not df.empty:
            return df
    except Exception:
        pass

    raise RuntimeError(f"모든 시세 수집 경로 실패: {symbol}")

print("[수집] KOSPI 및 후보 종목 일별 시세 수집 중...")
t0 = time.time()
KOSPI_DF = fetch_daily_ohlcv_resilient("KOSPI", start="2017-01-01")

PRICE_DATA = {}
failed_tickers = []
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = {ex.submit(fetch_daily_ohlcv_resilient, t, "2017-01-01"): t for t in sorted(NEEDED_TICKERS)}
    for fut in as_completed(futs):
        t = futs[fut]
        try:
            PRICE_DATA[t] = fut.result()
        except Exception as e:
            failed_tickers.append((t, str(e)))

print(f"[완료] {len(PRICE_DATA)}/{len(NEEDED_TICKERS)} 종목 + KOSPI 수집 완료 ({time.time() - t0:.1f}초)")
print(f"       KOSPI 일별 범위: {KOSPI_DF.index.min().date()} ~ {KOSPI_DF.index.max().date()} ({len(KOSPI_DF):,} 거래일)")
if failed_tickers:
    print(f"  [주의] 시세 수집 실패 {len(failed_tickers)}건: {failed_tickers[:5]}")

# 각 리밸런싱 시점(매년 4월 첫 영업일) 계산 및 전략 A / 전략 B 최종 종목 선정
TRADING_DAYS = KOSPI_DF.index

def first_trading_day_in_april(year: int) -> pd.Timestamp:
    cands = TRADING_DAYS[(TRADING_DAYS.year == year) & (TRADING_DAYS.month == 4)]
    if len(cands) == 0:
        raise ValueError(f"{year}년 4월 거래일이 없습니다.")
    return cands[0]

def compute_market_guards(ticker: str, as_of: pd.Timestamp) -> tuple[float | None, float | None, float | None]:
    '''as_of 당일(포함) 이전 일봉만 사용해 60일 일평균 거래대금, 52주 고점 대비 낙폭, 당일 종가 산출 (미래 누수 차단).'''
    df = PRICE_DATA.get(ticker)
    if df is None or df.empty:
        return None, None, None
    sub = df.loc[df.index <= as_of]
    if len(sub) < 20:
        return None, None, None
    cur_close = float(sub["close"].iloc[-1])
    turnover_60d = float((sub["close"] * sub["volume"]).tail(60).mean())
    w52 = sub.loc[sub.index >= as_of - pd.Timedelta(days=365)]
    high_52w = float(w52["high"].max()) if not w52.empty else cur_close
    dd_52w = (cur_close / high_52w - 1.0) if high_52w > 0 else 0.0
    return turnover_60d, dd_52w, cur_close

def rank_and_pick(cands: list[dict], val_key: str, top_n: int) -> list[dict]:
    if not cands:
        return []
    vals = [c[val_key] for c in cands]
    grs = [c["dps_g5"] if c["dps_g5"] is not None else -99.0 for c in cands]
    scored = []
    for c in cands:
        vr = 1 + sum(1 for v in vals if v > c[val_key])
        gr = 1 + sum(1 for g in grs if g > (c["dps_g5"] if c["dps_g5"] is not None else -99.0))
        c_copy = dict(c)
        c_copy["val_rank"] = vr
        c_copy["growth_rank"] = gr
        c_copy["rank_sum"] = vr + gr
        scored.append((vr + gr, -c[val_key], -c["spread"], c_copy))
    scored.sort(key=lambda x: x[:3])
    return [x[3] for x in scored[:top_n]]

PICKS_A, PICKS_B = {}, {}
ENRICHED_BY_FY = {}
FILTERED_OUT_B = []

for fy in range(CONFIG["START_FY"], CONFIG["END_FY"] + 1):
    rebal_dt = first_trading_day_in_april(fy + 1)
    enriched = []
    for c in YEARLY_POOL[fy]:
        t = c["ticker"]
        to60, dd52, px_rebal = compute_market_guards(t, rebal_dt)
        if px_rebal is None:
            continue
        item = dict(c)
        item["rebal_date"] = rebal_dt
        item["rebal_close"] = px_rebal
        item["turnover_60d"] = to60
        item["dd_52w"] = dd52
        item["rebal_yld"] = item["dps_y_adj"] / px_rebal if px_rebal > 0 else None  # 수정주가(분할·무상 보정)와 동일 단위로 맞춤

        # 전략 B 배당 함정 필터 판정
        reasons = []
        if item["rebal_yld"] is None or item["rebal_yld"] < CONFIG["B_MIN_YIELD"]:
            reasons.append(f"저배당({(item['rebal_yld'] or 0):.1%}<{CONFIG['B_MIN_YIELD']:.0%})")
        elif item["rebal_yld"] > CONFIG["B_MAX_YIELD"]:
            reasons.append(f"초고배당함정({item['rebal_yld']:.1%}>{CONFIG['B_MAX_YIELD']:.0%})")
        if item["dps_g5"] is None or item["dps_g5"] < CONFIG["B_MIN_DPS_CAGR_5Y"]:
            reasons.append(f"저성장({(item['dps_g5'] or 0):.1%}<{CONFIG['B_MIN_DPS_CAGR_5Y']:.0%})")
        if to60 is None or to60 < CONFIG["B_MIN_TURNOVER_60D"]:
            reasons.append(f"저유동성({(to60 or 0)/1e8:.1f}억)")
        if dd52 is not None and dd52 <= CONFIG["B_MAX_DD_52W_WITH_EPS_DROP"] and item["eps_down"]:
            reasons.append(f"급락+이익역성장(52주 {dd52:.1%})")
        item["b_pass"] = (len(reasons) == 0)
        item["b_reject_reason"] = ", ".join(reasons)
        enriched.append(item)

    ENRICHED_BY_FY[rebal_dt] = enriched
    picks_a = rank_and_pick(enriched, "val_raw", CONFIG["TOP_N"])
    picks_b = rank_and_pick([x for x in enriched if x["b_pass"]], "val_cap", CONFIG["TOP_N"])
    PICKS_A[rebal_dt] = picks_a
    PICKS_B[rebal_dt] = picks_b

    b_tickers = {x["ticker"] for x in picks_b}
    for x in picks_a:
        if not x["b_pass"]:
            FILTERED_OUT_B.append({
                "리밸런싱일": rebal_dt.strftime("%Y-%m-%d"),
                "종목명": x["name"], "티커": x["ticker"],
                "배당수익률": f"{(x['rebal_yld'] or 0):.2%}",
                "DPS 5년성장": f"{(x['dps_g5'] or 0):.1%}",
                "52주 낙폭": f"{(x['dd_52w'] or 0):.1%}",
                "60일 거래대금(억)": round((x["turnover_60d"] or 0) / 1e8, 1),
                "전략 B 제외 사유": x["b_reject_reason"],
            })

    print(f"  {rebal_dt.date()} (FY{fy}): 전략 A {len(picks_a):2d}종목 / 전략 B {len(picks_b):2d}종목 (전체 통과 A={len(enriched)}, B={sum(x['b_pass'] for x in enriched)})")

In [ ]:
# ==============================================================================
# 6. 일별 포트폴리오 백테스트 엔진 (배당 재투자 DRIP + 거래비용 + 세후 배당금 추적)
# ==============================================================================
# 종목별 연도별 실제 보정 DPS 매핑 (FY2017~FY2025)
ACT_DPS_MAP = {r["ticker"]: {int(y): float(v) for y, v in (r.get("act_dps") or {}).items()} for r in PIT_RECORDS}

def get_dps_for_calendar_year(ticker: str, year: int) -> tuple[float, bool]:
    '''결산연도 year의 실제 보정 DPS 반환. 아직 공시 전인 최신 연도는 직전 연도 DPS로 보수적 추정(추정 여부 표기).'''
    dmap = ACT_DPS_MAP.get(ticker) or {}
    if year in dmap:
        return dmap[year], False
    prev_years = [y for y in dmap if y < year]
    if prev_years:
        return dmap[max(prev_years)], True
    return 0.0, True

def run_daily_backtest(picks_by_date: dict[pd.Timestamp, list[dict]], label: str) -> dict:
    rebal_dates = sorted(picks_by_date.keys())
    start_dt = rebal_dates[0]
    cal = TRADING_DAYS[TRADING_DAYS >= start_dt]

    # 가격 패널 (전방 채움으로 휴장/거래정지일 직전 종가 유지)
    all_tickers = sorted({x["ticker"] for picks in picks_by_date.values() for x in picks})
    close_panel = pd.DataFrame(index=cal)
    for t in all_tickers:
        if t in PRICE_DATA:
            close_panel[t] = PRICE_DATA[t]["close"].reindex(cal).ffill()

    # 연도별 마지막 거래일 (12월 결산 배당 권리 확정일 기준)
    year_last_day = {}
    for y, grp in pd.Series(cal, index=cal).groupby(cal.year):
        if grp.iloc[-1].month == 12:
            year_last_day[y] = grp.iloc[-1]

    cash = float(CONFIG["INITIAL_CAPITAL"])
    shares = {}  # ticker -> float shares
    rebal_set = set(rebal_dates)
    fee = CONFIG["FEE_RATE"]
    div_tax = CONFIG["DIV_TAX_RATE"]
    reinvest = CONFIG["REINVEST_DIVIDENDS"]

    nav_series = []
    cash_div_by_year = {}
    div_yield_by_year = {}
    est_flag_by_year = {}
    turnover_log = []

    for dt in cal:
        # 현재 평가액 계산
        stock_val = 0.0
        for t, sh in shares.items():
            px = close_panel.at[dt, t] if t in close_panel.columns else np.nan
            if not np.isnan(px) and px > 0:
                stock_val += sh * px
        total_val = cash + stock_val

        # 리밸런싱일 처리 (매년 4월 첫 영업일)
        if dt in rebal_set:
            picks = [x for x in picks_by_date[dt] if x["ticker"] in close_panel.columns and pd.notna(close_panel.at[dt, x["ticker"]])]
            if picks:
                investable = total_val if reinvest else (stock_val if shares else total_val)
                target_w = {x["ticker"]: 1.0 / len(picks) for x in picks}
                cur_w = {}
                if investable > 0:
                    for t, sh in shares.items():
                        px = close_panel.at[dt, t]
                        if pd.notna(px) and px > 0:
                            cur_w[t] = (sh * px) / investable
                # 포지션 변화분 기준 거래비용 (매도 + 매수 각각 편도 수수료 적용)
                all_t = set(target_w) | set(cur_w)
                one_way_turnover = 0.5 * sum(abs(target_w.get(t, 0.0) - cur_w.get(t, 0.0)) for t in all_t)
                trade_cost = investable * (sum(abs(target_w.get(t, 0.0) - cur_w.get(t, 0.0)) for t in all_t)) * fee
                turnover_log.append((dt, one_way_turnover, trade_cost))

                net_investable = max(0.0, investable - trade_cost)
                new_shares = {}
                for t, w in target_w.items():
                    px = close_panel.at[dt, t]
                    new_shares[t] = (net_investable * w) / px
                shares = new_shares
                if reinvest:
                    cash = 0.0
                else:
                    # 배당 미재투자 시 누적 현금은 그대로 유지, 주식 원금만 거래비용 차감
                    cash = max(0.0, total_val - investable)
                total_val = cash + sum(shares[t] * close_panel.at[dt, t] for t in shares)

        # 연말 결산 배당 권리 확정 (12월 마지막 거래일 보유 주식 기준 세후 배당금 산출 -> 익년 4월 리밸런싱 시 재투자)
        if dt == year_last_day.get(dt.year) and shares:
            gross_div = 0.0
            any_est = False
            for t, sh in shares.items():
                px = close_panel.at[dt, t]
                dps, is_est = get_dps_for_calendar_year(t, dt.year)
                # 분할/증자 이상치 방어: 단일 종목 배당수익률 25% 초과는 캡
                if pd.notna(px) and px > 0 and dps / px > 0.25:
                    dps = px * 0.25
                gross_div += sh * dps
                any_est = any_est or is_est
            net_div = gross_div * (1.0 - div_tax)
            cash_div_by_year[dt.year] = net_div
            div_yield_by_year[dt.year] = net_div / total_val if total_val > 0 else 0.0
            est_flag_by_year[dt.year] = any_est
            cash += net_div
            total_val += net_div

        nav_series.append((dt, total_val))

    nav = pd.Series({d: v for d, v in nav_series}, name=label)
    dd = nav / nav.cummax() - 1.0
    return {
        "label": label,
        "nav": nav,
        "dd": dd,
        "annual_div": pd.Series(cash_div_by_year, dtype=float),
        "annual_div_yield": pd.Series(div_yield_by_year, dtype=float),
        "est_flag": est_flag_by_year,
        "turnover": turnover_log,
    }

RES_A = run_daily_backtest(PICKS_A, "전략 A (현행 정량 규칙)")
RES_B = run_daily_backtest(PICKS_B, "전략 B (배당 함정 보완안)")

# KOSPI 벤치마크 (동일 거래일 기준 정규화)
kospi_cal = KOSPI_DF["close"].reindex(RES_A["nav"].index).ffill()
KOSPI_NAV = kospi_cal / kospi_cal.iloc[0] * CONFIG["INITIAL_CAPITAL"]
KOSPI_DD = KOSPI_NAV / KOSPI_NAV.cummax() - 1.0

print(f"[백테스트 완료] 기간: {RES_A['nav'].index[0].date()} ~ {RES_A['nav'].index[-1].date()} ({len(RES_A['nav']):,} 거래일)")
print(f"  전략 A 최종 평가액: {RES_A['nav'].iloc[-1]:,.0f}원 ({RES_A['nav'].iloc[-1]/CONFIG['INITIAL_CAPITAL']-1:+.1%})")
print(f"  전략 B 최종 평가액: {RES_B['nav'].iloc[-1]:,.0f}원 ({RES_B['nav'].iloc[-1]/CONFIG['INITIAL_CAPITAL']-1:+.1%})")
print(f"  KOSPI  최종 평가액: {KOSPI_NAV.iloc[-1]:,.0f}원 ({KOSPI_NAV.iloc[-1]/CONFIG['INITIAL_CAPITAL']-1:+.1%})")

# ------------------------------------------------------------------------------
# 장기보유 전략 (실제 운용규칙 4절 A 매도 사유 준수: FAIL · 2년 연속 삭감 · 배당 중단 외 매도 금지)
# ------------------------------------------------------------------------------
ITEM_NAMES = {1: "1번 배당이력", 2: "2번 EPS성장", 3: "3번 현금꾸준함", 4: "4번 자본효율",
              5: "5번 환원수준", 6: "6번 자사주", 7: "7번 환원감당", 8: "8번 건전성"}

def check_sell_rule_pit(ticker: str, fy: int) -> tuple[bool, str]:
    '''docs/운용규칙.md 4절 A 정량 매도 사유 판정 (시점고정 PIT 데이터만 사용):
    - 1~8번 판정 FAIL 1건 이상
    - 정기배당 2년 연속 삭감 또는 배당 중단(DPS <= 0)
    (업황성 삭감 1회, 주가 하락, 고평가 WARN, 순위 이탈은 매도 사유 아님 -> 보유 유지!)'''
    rec = PIT_REC_MAP.get(ticker)
    if not rec:
        return True, "종목 자료 없음"
    fr = (rec.get("snapshots") or {}).get(str(fy))
    if not fr:
        return True, f"FY{fy} 공시 없음"
    dps = fr.get("dps_common") or {}
    d0, d1, d2 = _f(dps.get(str(fy))), _f(dps.get(str(fy - 1))), _f(dps.get(str(fy - 2)))
    reasons = []
    if d0 is None or d0 <= 0:
        reasons.append("배당 중단(DPS=0)")
    elif d1 is not None and d2 is not None and (d0 < d1 - min(0.5, abs(d1) * 1e-6)) and (d1 < d2 - min(0.5, abs(d2) * 1e-6)):
        reasons.append("정기배당 2년 연속 삭감")
    ev = evaluate_company_pit(rec, fy)
    if ev is not None and ev["fail_items"]:
        f_txt = "·".join(ITEM_NAMES.get(i, f"{i}번") for i in ev["fail_items"])
        reasons.append(f"FAIL({f_txt})")
    return (len(reasons) > 0), ", ".join(reasons)

def run_hold_until_sell_backtest(enriched_by_date: dict[pd.Timestamp, list[dict]], label: str) -> dict:
    '''초기 상위 TOP_N 매수 후, 매년 4월 정기 점검에서 매도 사유 발생 종목만 편출하고
    매도 대금 + 누적 세후 배당금(DRIP)만 신규 후보(빈자리) 또는 기존 상위 종목에 재투자.'''
    rebal_dates = sorted(enriched_by_date.keys())
    start_dt = rebal_dates[0]
    cal = TRADING_DAYS[TRADING_DAYS >= start_dt]

    # 전체 후보 풀 + 보유 가능 종목 일별 종가 패널
    all_tickers = sorted({x["ticker"] for enr in enriched_by_date.values() for x in enr})
    close_panel = pd.DataFrame(index=cal)
    for t in all_tickers:
        if t in PRICE_DATA:
            close_panel[t] = PRICE_DATA[t]["close"].reindex(cal).ffill()

    year_last_day = {}
    for y, grp in pd.Series(cal, index=cal).groupby(cal.year):
        if grp.iloc[-1].month == 12:
            year_last_day[y] = grp.iloc[-1]

    cash = float(CONFIG["INITIAL_CAPITAL"])
    shares = {}
    rebal_set = set(rebal_dates)
    fee = CONFIG["FEE_RATE"]
    div_tax = CONFIG["DIV_TAX_RATE"]
    reinvest = CONFIG["REINVEST_DIVIDENDS"]
    top_n = CONFIG["TOP_N"]

    nav_series = []
    cash_div_by_year = {}
    div_yield_by_year = {}
    est_flag_by_year = {}
    turnover_log = []
    sell_events = []
    yearly_holdings = {}

    for dt in cal:
        stock_val = sum(shares[t] * close_panel.at[dt, t] for t in shares
                        if t in close_panel.columns and pd.notna(close_panel.at[dt, t]))
        total_val = cash + stock_val

        if dt in rebal_set:
            fy = dt.year - 1
            enr = [x for x in enriched_by_date[dt] if x["ticker"] in close_panel.columns and pd.notna(close_panel.at[dt, x["ticker"]])]
            ranked_all = rank_and_pick(enr, "val_raw", len(enr))

            if not shares:
                # 첫 매수: 상위 TOP_N 동일가중 진입
                init_picks = ranked_all[:top_n]
                investable = total_val
                trade_cost = investable * fee
                net_inv = max(0.0, investable - trade_cost)
                for c in init_picks:
                    px = close_panel.at[dt, c["ticker"]]
                    shares[c["ticker"]] = (net_inv / len(init_picks)) / px
                cash = 0.0
                turnover_log.append((dt, 1.0, trade_cost))
            else:
                # 1) 보유 종목 매도 사유 점검 (FAIL / 2년 연속 삭감 / 배당 중단만 매도!)
                sold_cash = 0.0
                sell_cost = 0.0
                sold_names = []
                for t in list(shares.keys()):
                    px = close_panel.at[dt, t] if t in close_panel.columns else np.nan
                    if pd.isna(px) or px <= 0:
                        do_sell, why = True, "시세 결측"
                    else:
                        do_sell, why = check_sell_rule_pit(t, fy)
                    if do_sell:
                        val_t = shares[t] * (0.0 if pd.isna(px) else px)
                        cost_t = val_t * fee
                        sold_cash += max(0.0, val_t - cost_t)
                        sell_cost += cost_t
                        nm = PIT_REC_MAP[t]["name"] if t in PIT_REC_MAP else t
                        sold_names.append((t, nm, why))
                        del shares[t]

                # 2) 매도 대금 + (DRIP 켜짐 시) 누적 세후 배당금 재투입
                buy_budget = sold_cash + (cash if reinvest else 0.0)
                if reinvest:
                    cash = 0.0
                else:
                    # 배당 미재투자 시 매도 대금만 재투자, 누적 배당 현금은 그대로 둠
                    pass

                open_slots = max(0, top_n - len(shares))
                added_names = []
                buy_cost = 0.0
                if buy_budget > 0:
                    if open_slots > 0:
                        # 빈자리는 당시 재투자 순위 최상위 미보유 후보로 채움
                        new_cands = [c for c in ranked_all if c["ticker"] not in shares][:open_slots]
                        if new_cands:
                            per_slot = buy_budget / len(new_cands)
                            for c in new_cands:
                                px = close_panel.at[dt, c["ticker"]]
                                buy_cost += per_slot * fee
                                shares[c["ticker"]] = (per_slot * (1.0 - fee)) / px
                                added_names.append(c["name"])
                        else:
                            cash += buy_budget
                    else:
                        # 매도 0건(빈자리 없음): 누적 배당금을 보유 중인 원칙부합 상위 종목(없으면 현재 보유 동일비율)에 추가 매수
                        held_ranked = [c["ticker"] for c in ranked_all if c["ticker"] in shares][:3]
                        targets = held_ranked if held_ranked else list(shares.keys())
                        per_t = buy_budget / len(targets)
                        for t in targets:
                            px = close_panel.at[dt, t]
                            if pd.notna(px) and px > 0:
                                buy_cost += per_t * fee
                                shares[t] = shares.get(t, 0.0) + (per_t * (1.0 - fee)) / px

                one_way_to = (sold_cash / total_val) if total_val > 0 else 0.0
                turnover_log.append((dt, one_way_to, sell_cost + buy_cost))
                if sold_names:
                    for idx_s, (t, nm, why) in enumerate(sold_names):
                        rep_nm = added_names[idx_s] if idx_s < len(added_names) else "(현금보유)"
                        sell_events.append({
                            "점검일": dt.strftime("%Y-%m-%d"), "기준FY": f"FY{fy}",
                            "매도 종목": f"{nm}({t})", "매도 사유 (운용규칙 4절 A)": why,
                            "교체 편입 종목": rep_nm,
                        })

            yearly_holdings[dt] = [PIT_REC_MAP[t]["name"] if t in PIT_REC_MAP else t for t in shares]
            total_val = cash + sum(shares[t] * close_panel.at[dt, t] for t in shares if pd.notna(close_panel.at[dt, t]))

        if dt == year_last_day.get(dt.year) and shares:
            gross_div, any_est = 0.0, False
            for t, sh in shares.items():
                px = close_panel.at[dt, t]
                dps, is_est = get_dps_for_calendar_year(t, dt.year)
                if pd.notna(px) and px > 0 and dps / px > 0.25:
                    dps = px * 0.25
                gross_div += sh * dps
                any_est = any_est or is_est
            net_div = gross_div * (1.0 - div_tax)
            cash_div_by_year[dt.year] = net_div
            div_yield_by_year[dt.year] = net_div / total_val if total_val > 0 else 0.0
            est_flag_by_year[dt.year] = any_est
            cash += net_div
            total_val += net_div

        nav_series.append((dt, total_val))

    nav = pd.Series({d: v for d, v in nav_series}, name=label)
    dd = nav / nav.cummax() - 1.0
    return {
        "label": label, "nav": nav, "dd": dd,
        "annual_div": pd.Series(cash_div_by_year, dtype=float),
        "annual_div_yield": pd.Series(div_yield_by_year, dtype=float),
        "est_flag": est_flag_by_year, "turnover": turnover_log,
        "sell_events": sell_events, "yearly_holdings": yearly_holdings,
    }

RES_HOLD = run_hold_until_sell_backtest(ENRICHED_BY_FY, "전략 A-장기보유 (매도규칙)")
print(f"  전략 A-장기보유 평가액: {RES_HOLD['nav'].iloc[-1]:,.0f}원 ({RES_HOLD['nav'].iloc[-1]/CONFIG['INITIAL_CAPITAL']-1:+.1%}) | 총 매도 {len(RES_HOLD['sell_events'])}건")

In [ ]:
# ==============================================================================
# 7. 핵심 성과표 출력 (장기보유 매도규칙 vs 매년 리밸런싱 vs 함정 보완안 vs KOSPI)
# ==============================================================================
bt_years = sorted(set(RES_A["nav"].index.year))
RF_AVG = float(np.mean([RF_MAP.get(y, RF_MAP[max(RF_MAP)]) for y in bt_years]))

def calc_metrics(nav: pd.Series, dd: pd.Series, kospi_nav: pd.Series,
                 annual_div_yield: pd.Series | None = None, turnover_log: list | None = None) -> dict:
    days = (nav.index[-1] - nav.index[0]).days
    years = days / 365.25
    tot_ret = nav.iloc[-1] / nav.iloc[0] - 1.0
    cagr = (nav.iloc[-1] / nav.iloc[0]) ** (1.0 / years) - 1.0
    daily_ret = nav.pct_change().dropna()
    ann_vol = float(daily_ret.std() * np.sqrt(252))
    sharpe = (cagr - RF_AVG) / ann_vol if ann_vol > 0 else np.nan
    mdd = float(dd.min())
    mdd_dt = dd.idxmin().strftime("%Y-%m-%d")
    calmar = cagr / abs(mdd) if mdd < 0 else np.nan

    yr_ret = nav.groupby(nav.index.year).last().pct_change().dropna()
    k_yr_ret = kospi_nav.groupby(kospi_nav.index.year).last().pct_change().dropna()
    common_y = yr_ret.index.intersection(k_yr_ret.index)
    win_rate = float((yr_ret.loc[common_y] > k_yr_ret.loc[common_y]).mean()) if len(common_y) else np.nan

    avg_dy = float(annual_div_yield.mean()) if (annual_div_yield is not None and not annual_div_yield.empty) else np.nan
    # 초기 매수(첫해 100%) 제외한 연평균 편도 회전율
    avg_to = float(np.mean([x[1] for x in turnover_log[1:]])) if (turnover_log and len(turnover_log) > 1) else np.nan

    return {
        "최종 평가액 (1억 기준)": f"{nav.iloc[-1]:,.0f}원",
        "누적 수익률": f"{tot_ret:+.1%}",
        "CAGR (연평균)": f"{cagr:+.2%}",
        "MDD (최대낙폭)": f"{mdd:.1%} ({mdd_dt})",
        "연환산 변동성": f"{ann_vol:.1%}",
        f"샤프지수 (Rf={RF_AVG:.2%})": f"{sharpe:.2f}",
        "칼마지수 (CAGR/|MDD|)": f"{calmar:.2f}",
        "연간 승률 (vs KOSPI)": "-" if np.isnan(win_rate) else f"{win_rate:.0%}",
        "평균 세후 배당수익률": "-" if np.isnan(avg_dy) else f"{avg_dy:.2%}",
        "연평균 종목 교체율": "-" if np.isnan(avg_to) else f"{avg_to:.1%}",
    }

summary_df = pd.DataFrame({
    "전략 A-장기보유 (매도규칙)": calc_metrics(RES_HOLD["nav"], RES_HOLD["dd"], KOSPI_NAV, RES_HOLD["annual_div_yield"], RES_HOLD["turnover"]),
    "전략 A (매년 리밸런싱)": calc_metrics(RES_A["nav"], RES_A["dd"], KOSPI_NAV, RES_A["annual_div_yield"], RES_A["turnover"]),
    "전략 B (함정 보완안)": calc_metrics(RES_B["nav"], RES_B["dd"], KOSPI_NAV, RES_B["annual_div_yield"], RES_B["turnover"]),
    "KOSPI (가격지수 PR)": calc_metrics(KOSPI_NAV, KOSPI_DD, KOSPI_NAV, None, None),
})

print("■ 핵심 성과 비교표 (일별 수정주가 기준)")
display(summary_df)
if RES_HOLD["sell_events"]:
    print(f"\n■ [전략 A-장기보유] 실제 매도·교체 이력 (총 {len(RES_HOLD['sell_events'])}건 — FAIL·2년 연속 삭감만 매도)")
    display(pd.DataFrame(RES_HOLD["sell_events"]))

In [ ]:
# ==============================================================================
# 8. 시각화 대시보드 (고급 리서치 스타일 4종 차트 — 장기보유 매도규칙 포함)
# ==============================================================================
import io, base64

COLOR_H = "#059669"   # Emerald Green (전략 A-장기보유: 매도규칙 준수)
COLOR_A = "#1D4ED8"   # Deep Royal Blue (전략 A: 매년 리밸런싱)
COLOR_B = "#DC2626"   # Refined Crimson (전략 B: 함정 보완안)
COLOR_K = "#64748B"   # Cool Slate Gray (KOSPI)
BG_CARD = "#FFFFFF"
GRID_CLR = "#E2E8F0"

def style_axis(ax, title, ylabel="", pct_y=False):
    ax.set_facecolor(BG_CARD)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#CBD5E1")
    ax.spines["bottom"].set_color("#CBD5E1")
    ax.grid(True, axis="y", color=GRID_CLR, linestyle="--", linewidth=0.8, alpha=0.8)
    ax.grid(False, axis="x")
    ax.set_title(title, fontsize=11.5, fontweight="bold", color="#0F172A", loc="left", pad=10)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9.5, color="#475569")
    if pct_y:
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
    ax.tick_params(colors="#475569", labelsize=9)

fig, axes = plt.subplots(2, 2, figsize=(15.8, 10.5), facecolor="white")
fig.suptitle("한국 배당성장주 백테스트 성과 대시보드 (장기보유 vs 매년 리밸런싱 vs KOSPI)", fontsize=15, fontweight="bold", color="#0F172A", y=0.985)
fig.text(0.5, 0.952, f"기간: {RES_A['nav'].index[0].strftime('%Y-%m-%d')} ~ {RES_A['nav'].index[-1].strftime('%Y-%m-%d')}  |  초기 1억 원 · 상위 {CONFIG['TOP_N']}종목 · 배당 재투자(DRIP) · 비용/세금 반영",
         ha="center", fontsize=10, color="#64748B")

# (1) 일별 누적 수익률
ax1 = axes[0, 0]
ret_h = (RES_HOLD["nav"] / CONFIG["INITIAL_CAPITAL"] - 1.0) * 100
ret_a = (RES_A["nav"] / CONFIG["INITIAL_CAPITAL"] - 1.0) * 100
ret_b = (RES_B["nav"] / CONFIG["INITIAL_CAPITAL"] - 1.0) * 100
ret_k = (KOSPI_NAV / CONFIG["INITIAL_CAPITAL"] - 1.0) * 100

ax1.fill_between(ret_h.index, ret_h, 0, color=COLOR_H, alpha=0.06)
ax1.plot(ret_h.index, ret_h, label=f"전략 A-장기보유 ({ret_h.iloc[-1]:+.1f}%)", color=COLOR_H, lw=2.5)
ax1.plot(ret_a.index, ret_a, label=f"전략 A 리밸런싱 ({ret_a.iloc[-1]:+.1f}%)", color=COLOR_A, lw=2.0)
ax1.plot(ret_b.index, ret_b, label=f"전략 B 함정보완 ({ret_b.iloc[-1]:+.1f}%)", color=COLOR_B, lw=1.8, alpha=0.85)
ax1.plot(ret_k.index, ret_k, label=f"KOSPI PR ({ret_k.iloc[-1]:+.1f}%)", color=COLOR_K, lw=1.5, ls="--")
ax1.axhline(0, color="#94A3B8", lw=0.8)
style_axis(ax1, "① 일별 누적 수익률 추이", "누적 수익률 (%)", pct_y=True)
ax1.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="#E2E8F0", fontsize=8.8)

# (2) 일별 최대 낙폭 (Drawdown / MDD)
ax2 = axes[0, 1]
dd_h, dd_a, dd_b, dd_k = RES_HOLD["dd"] * 100, RES_A["dd"] * 100, RES_B["dd"] * 100, KOSPI_DD * 100
ax2.fill_between(dd_h.index, dd_h, 0, color=COLOR_H, alpha=0.14)
ax2.plot(dd_h.index, dd_h, label=f"장기보유 (MDD {dd_h.min():.1f}%)", color=COLOR_H, lw=2.1)
ax2.plot(dd_a.index, dd_a, label=f"전략 A (MDD {dd_a.min():.1f}%)", color=COLOR_A, lw=1.7)
ax2.plot(dd_b.index, dd_b, label=f"전략 B (MDD {dd_b.min():.1f}%)", color=COLOR_B, lw=1.5, alpha=0.85)
ax2.plot(dd_k.index, dd_k, label=f"KOSPI (MDD {dd_k.min():.1f}%)", color=COLOR_K, lw=1.4, ls="--")
style_axis(ax2, "② 일별 최대 낙폭 (Drawdown / MDD)", "고점 대비 낙폭 (%)", pct_y=True)
ax2.legend(loc="lower left", frameon=True, facecolor="white", edgecolor="#E2E8F0", fontsize=8.8)

# (3) 연도별 세후 배당금 수령액 추이 (장기보유 vs 전략 A 리밸런싱 vs 전략 B)
ax3 = axes[1, 0]
div_years = sorted(set(RES_HOLD["annual_div"].index) | set(RES_A["annual_div"].index))
x = np.arange(len(div_years))
w = 0.27
div_h_man = [RES_HOLD["annual_div"].get(y, 0.0) / 1e4 for y in div_years]
div_a_man = [RES_A["annual_div"].get(y, 0.0) / 1e4 for y in div_years]
div_b_man = [RES_B["annual_div"].get(y, 0.0) / 1e4 for y in div_years]
bh = ax3.bar(x - w, div_h_man, width=w, label="장기보유", color=COLOR_H, alpha=0.90)
ba = ax3.bar(x,     div_a_man, width=w, label="전략 A", color=COLOR_A, alpha=0.88)
bb = ax3.bar(x + w, div_b_man, width=w, label="전략 B", color=COLOR_B, alpha=0.82)
max_d = max(div_h_man + div_a_man + div_b_man + [1])
for bar in bh:
    h = bar.get_height()
    if h > 0:
        ax3.text(bar.get_x() + bar.get_width() / 2, h + max_d * 0.02, f"{h:,.0f}", ha="center", va="bottom", fontsize=8, fontweight="bold", color="#065F46")
ax3.set_xticks(x)
ax3.set_xticklabels([f"{y}년" for y in div_years])
ax3.set_ylim(0, max_d * 1.18)
style_axis(ax3, "③ 연도별 세후 배당금 수령액 (초기 1억 기준 · 숫자=장기보유)", "세후 배당금 (만 원)")
ax3.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="#E2E8F0", fontsize=8.8)

# (4) 리밸런싱 구간별 수익률 비교
ax4 = axes[1, 1]
rebal_dates = sorted(PICKS_A.keys())
periods = [(rebal_dates[i], rebal_dates[i + 1] if i + 1 < len(rebal_dates) else RES_A["nav"].index[-1])
           for i in range(len(rebal_dates))]
p_labels, pr_h, pr_a, pr_k = [], [], [], []
for d0, d1 in periods:
    p_labels.append(f"{d0.strftime('%y.%m')}~{d1.strftime('%y.%m')}")
    pr_h.append((RES_HOLD["nav"].loc[d1] / RES_HOLD["nav"].loc[d0] - 1.0) * 100)
    pr_a.append((RES_A["nav"].loc[d1] / RES_A["nav"].loc[d0] - 1.0) * 100)
    pr_k.append((KOSPI_NAV.loc[d1] / KOSPI_NAV.loc[d0] - 1.0) * 100)

px_idx = np.arange(len(p_labels))
bw = 0.26
ax4.bar(px_idx - bw, pr_h, width=bw, label="장기보유", color=COLOR_H, alpha=0.92)
ax4.bar(px_idx,      pr_a, width=bw, label="전략 A", color=COLOR_A, alpha=0.88)
ax4.bar(px_idx + bw, pr_k, width=bw, label="KOSPI",  color=COLOR_K, alpha=0.72)
ax4.axhline(0, color="#64748B", lw=0.8)
ax4.set_xticks(px_idx)
ax4.set_xticklabels(p_labels, rotation=16, fontsize=8.5)
style_axis(ax4, "④ 구간별 수익률 비교 (매년 4월 기준)", "구간 수익률 (%)", pct_y=True)
ax4.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="#E2E8F0", fontsize=8.8)

plt.tight_layout(rect=[0, 0, 1, 0.935])
_buf_main = io.BytesIO()
fig.savefig(_buf_main, format="png", dpi=160, bbox_inches="tight", facecolor="white")
CHART_MAIN_B64 = base64.b64encode(_buf_main.getvalue()).decode("ascii")
plt.show()

In [ ]:
# ==============================================================================
# 9. 연도별 선정 종목 리스트 및 전략 B 배당 함정 필터링 내역
# ==============================================================================
def build_picks_table(picks_dict: dict) -> pd.DataFrame:
    rows = []
    for dt in sorted(picks_dict.keys()):
        for rank_i, c in enumerate(picks_dict[dt], 1):
            rows.append({
                "리밸런싱일": dt.strftime("%Y-%m-%d"),
                "기준FY": f"FY{c['fy']}",
                "순위": rank_i,
                "종목명": c["name"],
                "티커": c["ticker"],
                "경로": c["path"],
                "배당수익률": f"{(c['rebal_yld'] or 0):.2%}",
                "DPS 5년성장": f"{(c['dps_g5'] or 0):.1%}",
                "밸류할인(원형)": f"{(c['val_raw'] or 0):+.1%}",
                "밸류할인(캡)": f"{(c['val_cap'] or 0):+.1%}",
                "52주 낙폭": f"{(c['dd_52w'] or 0):.1%}",
                "60일 거래대금(억)": round((c["turnover_60d"] or 0) / 1e8, 1),
            })
    return pd.DataFrame(rows)

df_picks_a = build_picks_table(PICKS_A)
df_picks_b = build_picks_table(PICKS_B)

print("■ [전략 B — 배당 함정 보완안] 연도별 최종 선정 종목 리스트")
display(df_picks_b)

print("\n■ [전략 A — 현행 규칙] 연도별 선정 종목 요약 (종목명)")
summary_by_yr = pd.DataFrame({
    "전략 A (현행)": {dt.strftime("%Y-%m-%d"): ", ".join(x["name"] for x in PICKS_A[dt]) for dt in sorted(PICKS_A)},
    "전략 B (보완)": {dt.strftime("%Y-%m-%d"): ", ".join(x["name"] for x in PICKS_B[dt]) for dt in sorted(PICKS_B)},
})
display(summary_by_yr)

if FILTERED_OUT_B:
    print(f"\n■ 전략 A 상위권 중 전략 B 함정 필터로 제외된 종목 ({len(FILTERED_OUT_B)}건)")
    display(pd.DataFrame(FILTERED_OUT_B))
else:
    print("\n■ 전략 A 상위권 중 전략 B 함정 필터로 제외된 종목 없음")

In [ ]:
# ==============================================================================
# 10. 개별 필터 효과 분해 (Ablation Test: 5개 조건을 전략 A에 1개씩만 적용해 검증)
# ==============================================================================
ABLATION_SPECS = [
    ("기준선: 전략 A (현행 규칙)", lambda x: True, "val_raw"),
    ("A + ① 배당수익률 2~12%", lambda x: x["rebal_yld"] is not None and CONFIG["B_MIN_YIELD"] <= x["rebal_yld"] <= CONFIG["B_MAX_YIELD"], "val_raw"),
    ("A + ② DPS 5년 성장 ≥ 3%", lambda x: x["dps_g5"] is not None and x["dps_g5"] >= CONFIG["B_MIN_DPS_CAGR_5Y"], "val_raw"),
    ("A + ③ 60일 거래대금 ≥ 10억", lambda x: x["turnover_60d"] is not None and x["turnover_60d"] >= CONFIG["B_MIN_TURNOVER_60D"], "val_raw"),
    ("A + ④ 급락(-35%)&EPS역성장 차단", lambda x: not (x["dd_52w"] is not None and x["dd_52w"] <= CONFIG["B_MAX_DD_52W_WITH_EPS_DROP"] and x["eps_down"]), "val_raw"),
    ("A + ⑤ 밸류 괴리율 ±50% 캡", lambda x: True, "val_cap"),
    ("전략 B (①~⑤ 전체 적용)", lambda x: x["b_pass"], "val_cap"),
]

def raw_metrics(res: dict) -> dict:
    nav, dd = res["nav"], res["dd"]
    years = (nav.index[-1] - nav.index[0]).days / 365.25
    tot = float(nav.iloc[-1] / nav.iloc[0] - 1.0)
    cagr = float((nav.iloc[-1] / nav.iloc[0]) ** (1.0 / years) - 1.0)
    mdd = float(dd.min())
    vol = float(nav.pct_change().dropna().std() * np.sqrt(252))
    sharpe = (cagr - RF_AVG) / vol if vol > 0 else np.nan
    calmar = cagr / abs(mdd) if mdd < 0 else np.nan
    avg_dy = float(res["annual_div_yield"].mean()) if not res["annual_div_yield"].empty else np.nan
    return {"end_val": float(nav.iloc[-1]), "tot": tot, "cagr": cagr, "mdd": mdd, "vol": vol,
            "sharpe": sharpe, "calmar": calmar, "dy": avg_dy}

ablation_rows = []
ablation_raw = {}
base_m = None

for name, pred, val_key in ABLATION_SPECS:
    picks = {dt: rank_and_pick([x for x in enr if pred(x)], val_key, CONFIG["TOP_N"])
             for dt, enr in ENRICHED_BY_FY.items()}
    res = run_daily_backtest(picks, name)
    m = raw_metrics(res)
    ablation_raw[name] = m
    if base_m is None:
        base_m = m
    d_cagr = m["cagr"] - base_m["cagr"]
    d_mdd = m["mdd"] - base_m["mdd"]       # 양수면 낙폭이 덜 빠짐(개선)
    d_sharpe = m["sharpe"] - base_m["sharpe"]
    if name.startswith("기준선"):
        judge = "기준선"
    elif d_sharpe >= 0.02 and d_cagr >= 0:
        judge = "✅ 개선"
    elif d_sharpe >= 0.01 or (d_mdd >= 0.02 and d_cagr > -0.01):
        judge = "🟡 방어 개선"
    elif abs(d_sharpe) < 0.01 and abs(d_cagr) < 0.005:
        judge = "➖ 차이 미미"
    else:
        judge = "❌ 열위(제외 권장)"

    ablation_rows.append({
        "전략 변형": name,
        "누적수익률": f"{m['tot']:+.1%}",
        "CAGR": f"{m['cagr']:+.2%}",
        "ΔCAGR": "-" if name.startswith("기준선") else f"{d_cagr*100:+.2f}%p",
        "MDD": f"{m['mdd']:.1%}",
        "ΔMDD": "-" if name.startswith("기준선") else f"{d_mdd*100:+.1f}%p",
        "샤프지수": f"{m['sharpe']:.2f}",
        "Δ샤프": "-" if name.startswith("기준선") else f"{d_sharpe:+.2f}",
        "평균 세후배당률": f"{m['dy']:.2%}",
        "판정": judge,
    })

ABLATION_DF = pd.DataFrame(ablation_rows)
print("■ 개별 필터 효과 분해표 (전략 A 기준선 대비 — 각 필터를 1개씩만 적용)")
display(ABLATION_DF)

# Ablation 비교 시각화 차트
fig_ab, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(15, 5.2), facecolor="white")
fig_ab.suptitle("개별 필터 효과 분해 (Ablation Test: 전략 A 대비 CAGR · MDD · 샤프지수)", fontsize=13, fontweight="bold")

labels = [r["전략 변형"] for r in ablation_rows][::-1]
cagrs = [ablation_raw[l]["cagr"] * 100 for l in labels]
mdds = [abs(ablation_raw[l]["mdd"]) * 100 for l in labels]
sharpes = [ablation_raw[l]["sharpe"] for l in labels]
y_pos = np.arange(len(labels))

bar_colors_c = ["#1D4ED8" if "기준선" in l else ("#059669" if ablation_raw[l]["cagr"] > base_m["cagr"] else "#DC2626") for l in labels]
ax_l.barh(y_pos, cagrs, color=bar_colors_c, alpha=0.85, height=0.58)
ax_l.axvline(base_m["cagr"] * 100, color="#1f77b4", ls="--", lw=1.2)
for yp, v in zip(y_pos, cagrs):
    ax_l.text(v + 0.3, yp, f"{v:.2f}%", va="center", fontsize=9)
ax_l.set_yticks(y_pos)
ax_l.set_yticklabels(labels, fontsize=9.5)
ax_l.set_xlabel("CAGR (연평균 수익률, %)")
ax_l.set_title("① 연평균 수익률 (CAGR) 비교", fontsize=11, fontweight="bold")

bar_colors_s = ["#1D4ED8" if "기준선" in l else ("#059669" if ablation_raw[l]["sharpe"] > base_m["sharpe"] else "#D97706") for l in labels]
ax_r.barh(y_pos, sharpes, color=bar_colors_s, alpha=0.85, height=0.58)
ax_r.axvline(base_m["sharpe"], color="#1f77b4", ls="--", lw=1.2)
for yp, s, md in zip(y_pos, sharpes, mdds):
    ax_r.text(s + 0.012, yp, f"Sharpe {s:.2f} (MDD -{md:.1f}%)", va="center", fontsize=8.8)
ax_r.set_yticks(y_pos)
ax_r.set_yticklabels(["" for _ in labels])
ax_r.set_xlabel("샤프지수 (높을수록 우수)")
ax_r.set_xlim(0, max(sharpes) * 1.45)
ax_r.set_title("② 위험 대비 성과 (샤프지수 & MDD)", fontsize=11, fontweight="bold")

for _ax in (ax_l, ax_r):
    _ax.spines["top"].set_visible(False)
    _ax.spines["right"].set_visible(False)
    _ax.grid(True, axis="x", color="#E2E8F0", ls="--", lw=0.8)
    _ax.grid(False, axis="y")
plt.tight_layout(rect=[0, 0, 1, 0.93])
_buf_ab = io.BytesIO()
fig_ab.savefig(_buf_ab, format="png", dpi=140, bbox_inches="tight", facecolor="white")
CHART_ABLATION_B64 = base64.b64encode(_buf_ab.getvalue()).decode("ascii")
plt.show()

In [ ]:
# ==============================================================================
# 11. 친절한 첫 페이지 요약 + 기관 리서치 스타일 PDF & HTML 리포트 생성 · 저장 · 다운로드
# ==============================================================================
from html import escape

def export_html_to_pdf(html_str: str, pdf_path: str, chart_b64_list: list[tuple[str, str]] | None = None) -> bool:
    """HTML -> 고화질 A4 PDF 변환 (WeasyPrint 우선, 실패 시 Matplotlib 멀티페이지 PDF 폴백)."""
    try:
        from weasyprint import HTML as WP_HTML
        WP_HTML(string=html_str).write_pdf(pdf_path)
        return True
    except Exception as e:
        print(f"  [참고] WeasyPrint 변환 건너뜀({type(e).__name__}) -> Matplotlib PDF 폴백 생성 중...")
    try:
        from matplotlib.backends.backend_pdf import PdfPages
        import matplotlib.image as mpimg
        with PdfPages(pdf_path) as pdf:
            for title, b64 in (chart_b64_list or []):
                img = mpimg.imread(io.BytesIO(base64.b64decode(b64)), format="png")
                fig_p = plt.figure(figsize=(11.69, 8.27))
                plt.imshow(img)
                plt.axis("off")
                plt.title(title, fontsize=13, fontweight="bold", pad=12)
                plt.tight_layout()
                pdf.savefig(fig_p, dpi=150)
                plt.close(fig_p)
        return True
    except Exception as e2:
        print(f"  [주의] PDF 생성 실패: {e2}")
        return False

KST = timezone(timedelta(hours=9))
run_stamp = datetime.now(KST).strftime("%Y-%m-%d_%H%M%S")   # 실행할 때마다 KST 시각 기준 새 파일 생성 (덮어쓰기 방지!)
as_of_str = RES_A["nav"].index[-1].strftime("%Y-%m-%d")
start_str = RES_A["nav"].index[0].strftime("%Y-%m-%d")
bt_yrs_num = (RES_A["nav"].index[-1] - RES_A["nav"].index[0]).days / 365.25

hold_m = raw_metrics(RES_HOLD)
kospi_m = raw_metrics({"nav": KOSPI_NAV, "dd": KOSPI_DD, "annual_div_yield": pd.Series(dtype=float)})
b_all_m = raw_metrics(RES_B)
hold_avg_to = float(np.mean([x[1] for x in RES_HOLD["turnover"][1:]])) if len(RES_HOLD["turnover"]) > 1 else 0.0
a_avg_to = float(np.mean([x[1] for x in RES_A["turnover"][1:]])) if len(RES_A["turnover"]) > 1 else 0.0

# 5개 개별 필터만 분리해 '도움이 된 필터' vs '오히려 깎은 필터'를 쉬운 말로 자동 분류
single_filters = [k for k in ablation_raw if k.startswith("A + ")]
helpful = [k.replace("A + ", "") for k in single_filters if (ablation_raw[k]["sharpe"] - base_m["sharpe"]) >= 0.015]
harmful = [k.replace("A + ", "") for k in single_filters if (ablation_raw[k]["sharpe"] - base_m["sharpe"]) <= -0.015]
neutral = [k.replace("A + ", "") for k in single_filters if abs(ablation_raw[k]["sharpe"] - base_m["sharpe"]) < 0.015]

best_name = max(ablation_raw.keys(), key=lambda k: ablation_raw[k]["sharpe"])
best_m = ablation_raw[best_name]

# 쉬운 말 최종 결론 3줄 생성
diff_k_tot = (base_m["tot"] - kospi_m["tot"]) * 100
diff_k_mdd = (base_m["mdd"] - kospi_m["mdd"]) * 100   # 양수면 덜 빠짐
verdict_1 = (
    f"<strong>① 현행 전략(전략 A)의 실력:</strong> 초기 1억 원을 투자했다면 약 {bt_yrs_num:.1f}년 뒤 "
    f"<strong>{base_m['end_val']/1e8:.2f}억 원({base_m['tot']:+.1%}, 연평균 {base_m['cagr']:+.1%})</strong>이 되어, "
    f"같은 기간 코스피({kospi_m['end_val']/1e8:.2f}억 원, {kospi_m['tot']:+.1%}) 대비 "
    f"<strong>{diff_k_tot:+.1f}%p {'앞섰고' if diff_k_tot >= 0 else '뒤졌고'}</strong>, "
    f"최대 낙폭(MDD)도 <strong>{base_m['mdd']:.1%}</strong>로 코스피({kospi_m['mdd']:.1%})보다 "
    f"<strong>{abs(diff_k_mdd):.1f}%p {'덜 빠졌습니다' if diff_k_mdd >= 0 else '더 빠졌습니다'}</strong>."
)

parts_f = []
if helpful:
    parts_f.append(f"성과를 끌어올린 조건은 <strong>{escape(', '.join(helpful))}</strong>")
if harmful:
    parts_f.append(f"오히려 성과를 깎은 조건은 <strong>{escape(', '.join(harmful))}</strong>")
if neutral:
    parts_f.append(f"차이가 거의 없는 조건은 <strong>{escape(', '.join(neutral))}</strong>")
verdict_2 = "<strong>② 보완 필터 5개 개별 검증:</strong> " + (" / ".join(parts_f) if parts_f else "기준선과 유사") + "입니다."

if best_name.startswith("기준선"):
    verdict_3 = (
        "<strong>③ 실전 권고:</strong> 굳이 복잡한 추가 필터를 덧대기보다 <strong>지금의 현행 규칙(전략 A) 그대로 유지</strong>하는 것이 "
        "과적합 위험 없이 가장 안정적이고 우수합니다."
    )
else:
    verdict_3 = (
        f"<strong>③ 실전 권고:</strong> 이번 {len(PICKS_A)}년(리밸런싱 {len(PICKS_A)}회) 표본에서는 <strong>{escape(best_name)}</strong>의 위험 대비 성과(샤프 {best_m['sharpe']:.2f})가 가장 높았습니다. "
        f"다만 6년·연간 {CONFIG['TOP_N']}종목 표본은 1~2개 종목의 대박·급락에도 순위가 바뀔 수 있으므로, "
        f"전체 조건을 한꺼번에 바꾸기보다 <strong>실제 개선 근거가 명확한 조건 1~2개만 선별 참고</strong>하는 것을 권장합니다."
    )

avg_div_man_a = RES_A["annual_div"].mean() / 1e4 if not RES_A["annual_div"].empty else 0.0
last_div_man_a = RES_A["annual_div"].iloc[-1] / 1e4 if not RES_A["annual_div"].empty else 0.0
last_div_yr = int(RES_A["annual_div"].index[-1]) if not RES_A["annual_div"].empty else 2025

def badge_html(val: str) -> str:
    s = str(val)
    if "개선" in s and "방어" not in s:
        return f'<span class="pill pill-green">{escape(s)}</span>'
    if "방어" in s:
        return f'<span class="pill pill-amber">{escape(s)}</span>'
    if "열위" in s:
        return f'<span class="pill pill-red">{escape(s)}</span>'
    if "기준선" in s:
        return f'<span class="pill pill-blue">{escape(s)}</span>'
    return escape(s)

def render_styled_table(df: pd.DataFrame, index: bool = False, badge_cols: tuple = ()) -> str:
    d = df.copy()
    if index:
        d = d.reset_index()
        d = d.rename(columns={d.columns[0]: "항목"})
    ths = "".join(f"<th>{escape(str(c))}</th>" for c in d.columns)
    trs = []
    for _, row in d.iterrows():
        tds = []
        for c in d.columns:
            val = row[c]
            cell_html = badge_html(val) if c in badge_cols else escape(str(val))
            tds.append(f"<td>{cell_html}</td>")
        trs.append("<tr>" + "".join(tds) + "</tr>")
    return f'<table class="tbl"><thead><tr>{ths}</tr></thead><tbody>{"".join(trs)}</tbody></table>'

font_css = ""
if os.path.exists("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"):
    font_css = '''
    @font-face { font-family: 'NanumPDF'; src: url('file:///usr/share/fonts/truetype/nanum/NanumGothic.ttf'); font-weight: 400; }
    @font-face { font-family: 'NanumPDF'; src: url('file:///usr/share/fonts/truetype/nanum/NanumGothicBold.ttf'); font-weight: 700; }
    '''

html_doc = f'''<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>한국 배당성장주 백테스트 리포트 ({as_of_str})</title>
<style>
  {font_css}
  @page {{
    size: A4 portrait;
    margin: 12mm 12mm 14mm 12mm;
    @bottom-left {{ content: "한국 배당성장주 전략 검증 리포트 ({as_of_str})"; font-family: 'NanumPDF', sans-serif; font-size: 7.5pt; color: #94a3b8; }}
    @bottom-right {{ content: "Page " counter(page) " / " counter(pages); font-family: 'NanumPDF', sans-serif; font-size: 7.5pt; color: #94a3b8; }}
  }}
  :root {{
    --navy:#0B1F3A; --navy2:#163A6B; --gold:#C89B3C; --ink:#0F172A; --sub:#475569;
    --muted:#64748B; --bg:#F8FAFC; --card:#FFFFFF; --line:#E2E8F0; --blue:#1D4ED8;
    --green:#059669; --red:#DC2626; --amber:#D97706;
  }}
  * {{ box-sizing:border-box; }}
  body {{
    margin:0; padding:14px; background:var(--bg); color:var(--ink);
    font-family:'NanumPDF','NanumGothic',-apple-system,BlinkMacSystemFont,'Malgun Gothic',sans-serif;
    font-size:12.5px; line-height:1.6; font-variant-numeric:tabular-nums;
  }}
  .wrap {{ max-width:980px; margin:0 auto; }}
  .masthead {{
    background:linear-gradient(135deg, var(--navy) 0%, var(--navy2) 100%);
    color:#fff; padding:20px 24px; border-radius:12px; margin-bottom:13px;
    border-bottom:4px solid var(--gold);
  }}
  .eyebrow {{ font-size:9px; letter-spacing:1.4px; color:#FDE68A; font-weight:700; text-transform:uppercase; margin-bottom:4px; }}
  .masthead h1 {{ margin:0 0 5px; font-size:20px; font-weight:700; letter-spacing:-0.3px; }}
  .masthead .subtitle {{ margin:0; font-size:11.5px; color:#CBD5E1; }}
  .meta-row {{ display:flex; flex-wrap:wrap; gap:6px; margin-top:10px; }}
  .chip {{ background:rgba(255,255,255,0.13); border:1px solid rgba(255,255,255,0.22); padding:2.5px 9px; border-radius:999px; font-size:10px; color:#F1F5F9; }}

  .kpis {{ display:grid; grid-template-columns:repeat(4, 1fr); gap:10px; margin-bottom:13px; break-inside:avoid; }}
  .kpi {{ background:var(--card); border:1px solid var(--line); border-top:3px solid var(--blue); border-radius:10px; padding:11px 13px; }}
  .kpi.kpi-green {{ border-top-color:var(--green); }}
  .kpi.kpi-slate {{ border-top-color:#64748B; }}
  .kpi.kpi-gold {{ border-top-color:var(--gold); }}
  .kpi .lab {{ font-size:10px; color:var(--muted); font-weight:700; margin-bottom:2px; }}
  .kpi .val {{ font-size:18px; font-weight:700; color:var(--ink); letter-spacing:-0.4px; }}
  .kpi .sub {{ font-size:9.8px; color:var(--sub); margin-top:3px; line-height:1.4; }}

  .card {{ background:var(--card); border:1px solid var(--line); border-radius:10px; padding:14px 16px; margin-bottom:11px; break-inside:avoid; }}
  .card.allow-break {{ break-inside:auto; }}
  .sec-title {{ display:flex; align-items:center; gap:8px; margin:0 0 9px; font-size:13.5px; font-weight:700; color:var(--navy); }}
  .sec-num {{ display:inline-block; background:var(--navy); color:#fff; font-size:9.5px; padding:2px 7px; border-radius:5px; }}
  .sec-desc {{ font-size:11px; color:var(--sub); margin:-4px 0 9px; }}

  .exec-box {{ background:#F0F7FF; border:1px solid #BFDBFE; border-left:4px solid var(--blue); border-radius:8px; padding:12px 15px; margin-bottom:12px; }}
  .exec-box ul {{ margin:0; padding-left:18px; }}
  .exec-box li {{ margin-bottom:6px; font-size:11.5px; color:#1E293B; }}
  .exec-box li:last-child {{ margin-bottom:0; }}

  .warn-box {{ background:#FFFBEB; border:1px solid #FDE68A; border-left:4px solid var(--amber); border-radius:8px; padding:11px 14px; margin-bottom:11px; font-size:10.8px; color:#78350F; }}
  .warn-box b {{ color:#92400E; }}
  .warn-box ul {{ margin:5px 0 0; padding-left:17px; }}
  .warn-box li {{ margin-bottom:3px; }}

  .gloss {{ display:grid; grid-template-columns:repeat(2, 1fr); gap:7px 14px; font-size:10.3px; color:#334155; background:#F8FAFC; border:1px solid var(--line); border-radius:8px; padding:10px 12px; }}
  .gloss b {{ color:var(--navy); }}

  .scroll {{ overflow-x:auto; }}
  table.tbl {{ width:100%; border-collapse:collapse; font-size:10.2px; }}
  table.tbl thead {{ display:table-header-group; }}
  table.tbl tr {{ break-inside:avoid; }}
  table.tbl th {{ background:#F1F5F9; color:#334155; font-weight:700; padding:6.5px 6px; border-top:1.5px solid var(--navy); border-bottom:1px solid #CBD5E1; text-align:right; }}
  table.tbl td {{ padding:6px 6px; border-bottom:1px solid #EEF2F6; text-align:right; color:#1E293B; }}
  table.tbl tbody tr:nth-child(even) {{ background:#FAFCFF; }}
  table.tbl th:first-child, table.tbl td:first-child {{ text-align:left; font-weight:700; color:#0F172A; }}

  .pill {{ display:inline-block; padding:2px 7px; border-radius:999px; font-size:9px; font-weight:700; white-space:nowrap; }}
  .pill-green {{ background:#DCFCE7; color:#166534; }}
  .pill-amber {{ background:#FEF3C7; color:#92400E; }}
  .pill-red {{ background:#FEE2E2; color:#991B1B; }}
  .pill-blue {{ background:#DBEAFE; color:#1E40AF; }}

  img.chart {{ width:100%; height:auto; display:block; border-radius:6px; }}
  .page-break {{ page-break-before:always; break-before:page; }}
  .foot {{ font-size:9px; color:var(--muted); border-top:1px solid var(--line); padding-top:7px; margin-top:6px; }}

  @media (max-width: 720px) {{
    .kpis {{ grid-template-columns:repeat(2, 1fr); }}
    .gloss {{ grid-template-columns:1fr; }}
  }}
  @media print {{
    body {{ background:#fff; padding:0; font-size:9pt; }}
    .wrap {{ max-width:none; }}
    .scroll {{ overflow:visible; }}
    table.tbl {{ font-size:7.6pt; }}
    table.tbl th, table.tbl td {{ padding:4.5px 4px; }}
    .kpi .val {{ font-size:14.5pt; }}
  }}
</style>
</head>
<body>
<div class="wrap">
  <!-- ======================== PAGE 1: 친절한 최종 요약 & 결론 ======================== -->
  <div class="masthead">
    <div class="eyebrow">Executive Summary · Korean Dividend Growth Backtest</div>
    <h1>한국 배당성장주 전략 검증 리포트</h1>
    <p class="subtitle">1억 원을 매년 4월 선정 종목에 동일가중 투자·배당 재투자했을 때의 과거 성과와 5가지 보완 필터 검증 결과</p>
    <div class="meta-row">
      <span class="chip">검증 기간 {start_str} ~ {as_of_str} ({bt_yrs_num:.1f}년)</span>
      <span class="chip">대상 한국 상장 374개사</span>
      <span class="chip">매년 4월 상위 {CONFIG['TOP_N']}종목 리밸런싱</span>
      <span class="chip">매매비용 {CONFIG['FEE_RATE']:.2%} · 배당세 {CONFIG['DIV_TAX_RATE']:.1%} 반영</span>
    </div>
  </div>

  <div class="kpis">
    <div class="kpi">
      <div class="lab">전략 A (내 현행 규칙)</div>
      <div class="val" style="color:#1D4ED8">{base_m['end_val']/1e8:.2f}억 원</div>
      <div class="sub">누적 <b>{base_m['tot']:+.1%}</b> (연 {base_m['cagr']:+.1%})<br>최대낙폭 {base_m['mdd']:.1%} · 샤프 {base_m['sharpe']:.2f}</div>
    </div>
    <div class="kpi kpi-green">
      <div class="lab">전략 A-장기보유 (매도규칙)</div>
      <div class="val" style="color:#059669">{hold_m['end_val']/1e8:.2f}억 원</div>
      <div class="sub">누적 <b>{hold_m['tot']:+.1%}</b> (연 {hold_m['cagr']:+.1%})<br>최대낙폭 {hold_m['mdd']:.1%} · 샤프 {hold_m['sharpe']:.2f} · 교체율 {hold_avg_to:.0%}/년</div>
    </div>
    <div class="kpi kpi-slate">
      <div class="lab">비교 기준: 코스피(PR)</div>
      <div class="val" style="color:#475569">{kospi_m['end_val']/1e8:.2f}억 원</div>
      <div class="sub">누적 <b>{kospi_m['tot']:+.1%}</b> (연 {kospi_m['cagr']:+.1%})<br>최대낙폭 {kospi_m['mdd']:.1%} · 샤프 {kospi_m['sharpe']:.2f}</div>
    </div>
    <div class="kpi kpi-gold">
      <div class="lab">전략 A 연간 세후 배당금</div>
      <div class="val" style="color:#B45309">연 {avg_div_man_a:,.0f}만 원</div>
      <div class="sub">평균 세후 배당률 {base_m['dy']:.2%}<br>{last_div_yr}년 수령액 {last_div_man_a:,.0f}만 원</div>
    </div>
  </div>

  <div class="card">
    <div class="sec-title"><span class="sec-num">요약</span>30초 만에 읽는 최종 결론</div>
    <div class="exec-box">
      <ul>
        <li>{verdict_1}</li>
        <li><strong>①-2 실제 장기보유(매도규칙 준수) 결과:</strong> 매년 종목을 갈아타지 않고 <b>FAIL 또는 배당 2년 연속 삭감·중단 시에만 매도</b>했을 때, 1억 원은 <strong>{hold_m["end_val"]/1e8:.2f}억 원({hold_m["tot"]:+.1%}, 연 {hold_m["cagr"]:+.1%}, MDD {hold_m["mdd"]:.1%}, 샤프 {hold_m["sharpe"]:.2f})</strong>을 기록했습니다. 연평균 교체율은 <b>{hold_avg_to:.1%}</b>(매년 리밸런싱 {a_avg_to:.1%})로 매매 부담과 세금·비용이 크게 줄었습니다(총 매도 {len(RES_HOLD["sell_events"])}건).</li>
        <li>{verdict_2}</li>
        <li>{verdict_3}</li>
      </ul>
    </div>
    <div class="gloss">
      <div><b>CAGR (연평균 수익률):</b> 매년 복리로 평균 몇 %씩 불어났는지</div>
      <div><b>MDD (최대 낙폭):</b> 가장 높았던 시점 대비 최악의 하락 폭 (작을수록 안전)</div>
      <div><b>샤프지수 (Sharpe):</b> 감수한 출렁임 대비 수익 효율 (높을수록 우수)</div>
      <div><b>배당 재투자 (DRIP):</b> 매년 받은 세후 배당금을 다음 4월 리밸런싱에 재투입</div>
    </div>
  </div>

  <div class="warn-box">
    <b>⚠️ 이 백테스트를 읽을 때 반드시 알고 있어야 할 3가지 한계 (편향 체크)</b>
    <ul>
      <li><b>생존 편향(Survivorship Bias) 존재:</b> 374개사는 현재 상장 유지 중인 기업 목록이라, 과거 {len(PICKS_A)}여 년 사이 상장폐지·합병된 부실 기업이 빠져 있습니다. 따라서 절대 수익률(+200%대)은 실제보다 다소 높게 측정될 수 있습니다.</li>
      <li><b>미래 정보 누출(Look-Ahead Bias) 차단 방식:</b> 종목 선정 시 재무·배당·52주 낙폭·60일 거래대금은 모두 매수 시점(매년 4월) 이전 공시·시세만 사용했습니다. 단, 이후 액면분할·무상증자가 있었던 종목의 배당수익률은 수정주가와 주당 단위를 맞추기 위해 분할 비율만 보정했습니다.</li>
      <li><b>소수 표본의 우연(과적합) 주의:</b> {len(PICKS_A)}년(리밸런싱 {len(PICKS_A)}회)·연간 {CONFIG['TOP_N']}종목 표본에서는 <b>단 1~2개 종목의 대박/급락으로도 전략 A와 B의 순위가 바뀔 수 있습니다.</b> 백테스트 우승 조합을 맹신하지 말고, 전략의 논리적 타당성과 함께 해석해야 합니다.</li>
    </ul>
  </div>

  <!-- ======================== PAGE 2: 성과표 & 5개 필터 분해표 ======================== -->
  <div class="card page-break">
    <div class="sec-title"><span class="sec-num">01</span>5개 보완 필터 개별 효과 분해 (Ablation Test)</div>
    <div class="sec-desc">현행 전략 A(기준선)에 보완 조건 5개를 <b>한 번에 1개씩만</b> 얹어, 어떤 조건이 진짜 도움이 되고 어떤 조건이 방해되는지 비교한 표입니다. (Δ는 전략 A 대비 변화)</div>
    <div class="scroll">{render_styled_table(ABLATION_DF, index=False, badge_cols=("판정",))}</div>
  </div>

  <div class="card">
    <div class="sec-title"><span class="sec-num">02</span>핵심 성과 비교표 (전략 A vs 전략 B 전체 vs 코스피)</div>
    <div class="sec-desc">일별 수정주가 기준 성과입니다. 코스피(PR)는 배당이 포함되지 않은 가격지수이므로, 배당 재투자 포트폴리오와 비교 시 연 약 1.5~2.0%p 배당 차이를 감안해 보셔야 합니다.</div>
    <div class="scroll">{render_styled_table(summary_df, index=True)}</div>
  </div>

  <!-- ======================== PAGE 3: 차트 대시보드 ======================== -->
  <div class="card page-break">
    <div class="sec-title"><span class="sec-num">03</span>일별 백테스트 성과 대시보드</div>
    <img class="chart" src="data:image/png;base64,{CHART_MAIN_B64}" alt="성과 대시보드">
  </div>

  <div class="card">
    <div class="sec-title"><span class="sec-num">04</span>개별 필터 분해(Ablation) 비교 차트</div>
    <img class="chart" src="data:image/png;base64,{CHART_ABLATION_B64}" alt="필터 분해 차트">
  </div>

  <!-- ======================== PAGE 4+: 연도별 선정 종목 상세 ======================== -->
  {f'<div class="card allow-break"><div class="sec-title"><span class="sec-num">05-A</span>장기보유 전략 실제 매도·교체 이력 (총 {len(RES_HOLD["sell_events"])}건)</div><div class="sec-desc">운용규칙 4절 A(1~8번 FAIL 또는 정기배당 2년 연속 삭감·중단)에 걸린 종목만 편출했고, 그 외에는 주가 하락·순위 이탈과 무관하게 계속 보유했습니다.</div><div class="scroll">{render_styled_table(pd.DataFrame(RES_HOLD["sell_events"]), index=False)}</div></div>' if RES_HOLD["sell_events"] else ''}

  <div class="card page-break allow-break">
    <div class="sec-title"><span class="sec-num">05</span>연도별 선정 종목 한눈에 보기 (전략 A vs 전략 B)</div>
    <div class="scroll">{render_styled_table(summary_by_yr, index=True)}</div>
  </div>

  <div class="card allow-break">
    <div class="sec-title"><span class="sec-num">06</span>전략 A (현행 규칙) 연도별 선정 종목 상세</div>
    <div class="scroll">{render_styled_table(df_picks_a, index=False)}</div>
  </div>

  {f'<div class="card allow-break"><div class="sec-title"><span class="sec-num">07</span>전략 B 필터로 제외된 전략 A 상위 종목 ({len(FILTERED_OUT_B)}건)</div><div class="sec-desc">전략 A에서는 상위권이었지만 전략 B의 배당 함정 필터에 걸려 제외된 종목과 이유입니다.</div><div class="scroll">{render_styled_table(pd.DataFrame(FILTERED_OUT_B), index=False)}</div></div>' if FILTERED_OUT_B else ''}

  <div class="foot">
    ※ 본 리포트는 한국 상장사 공개 공시(OpenDART 집계) 및 공개 시세 기반의 전략 연구 자료이며, 특정 종목에 대한 투자 권유가 아닙니다.
  </div>
</div>
</body>
</html>'''

report_html = f"backtest_report_{run_stamp}.html"
report_pdf = f"backtest_report_{run_stamp}.pdf"
Path(report_html).write_text(html_doc, encoding="utf-8")
pdf_ok = export_html_to_pdf(html_doc, report_pdf, [
    ("성과 대시보드", CHART_MAIN_B64),
    ("개별 필터 효과 분해(Ablation)", CHART_ABLATION_B64),
])
ABLATION_DF.to_csv(f"ablation_{run_stamp}.csv", index=False, encoding="utf-8-sig")
df_picks_a.to_csv(f"picks_strategy_A_{run_stamp}.csv", index=False, encoding="utf-8-sig")

print(f"[리포트 생성 완료] PDF={'성공 (' + str(round(Path(report_pdf).stat().st_size/1024, 1)) + ' KB)' if pdf_ok and Path(report_pdf).exists() else '실패'} | HTML={Path(report_html).stat().st_size/1024:,.1f} KB")

if IN_COLAB and CONFIG.get("SAVE_REPORT_TO_DRIVE"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        d_rep = Path(CONFIG["DRIVE_REPORT_DIR"])
        d_rep.mkdir(parents=True, exist_ok=True)
        if pdf_ok and Path(report_pdf).exists():
            (d_rep / report_pdf).write_bytes(Path(report_pdf).read_bytes())
        (d_rep / report_html).write_text(html_doc, encoding="utf-8")
        ABLATION_DF.to_csv(d_rep / f"ablation_{run_stamp}.csv", index=False, encoding="utf-8-sig")
        df_picks_a.to_csv(d_rep / f"picks_strategy_A_{run_stamp}.csv", index=False, encoding="utf-8-sig")
        print(f"[Google Drive 자동 저장 완료] {d_rep / report_pdf}")
    except Exception as e:
        print(f"[Drive 저장 건너뜀] ({type(e).__name__})")

if IN_COLAB:
    from google.colab import files
    target_dl = report_pdf if (pdf_ok and Path(report_pdf).exists()) else report_html
    print(f"[다운로드 시작] {target_dl}")
    files.download(target_dl)